# 02 · Model Development & Experiment Tracking

This notebook is a thin wrapper around `src.train.train(...)` so results are always
consistent between notebook runs, CI runs, and Docker builds.

It also demonstrates how to browse the MLflow UI (`mlflow ui --backend-store-uri ./mlruns`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
PROJECT_ROOT = ROOT if (ROOT / 'src').is_dir() else ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from src import config
from src.train import train
print('MLflow tracking URI :', config.MLFLOW_TRACKING_URI)
print('MLflow experiment   :', config.MLFLOW_EXPERIMENT)

## 1. Train all candidate models with GridSearchCV

> Set `fast=True` for a quick pass; set `fast=False` for the full grid used in the report.

In [ ]:
summary = train(fast=False)
print(json.dumps({'best_model': summary['best_model'], 'best_roc_auc': summary['best_roc_auc']}, indent=2))

## 2. Compare candidate models

In [ ]:
rows = []
for name, r in summary['candidates'].items():
    row = {'model': name, **r['metrics']}
    rows.append(row)
table = pd.DataFrame(rows).sort_values('roc_auc', ascending=False).reset_index(drop=True)
table

## 3. Browsing MLflow runs

```bash
cd /path/to/repo
mlflow ui --backend-store-uri ./mlruns
# open http://127.0.0.1:5000
```

Screenshots of the MLflow UI (experiments list, run detail, artifact panel, model registry)
are stored in `screenshots/` and referenced in `docs/report.md`.